In [1]:
import pandas as pd
import numpy as np

print( "Libraries imported succesfully")

Libraries imported succesfully


In [2]:
ktas = pd.read_csv('00_raw/ktas_original.csv', encoding='latin-1', sep=';')

ktas.head()

,Group,Sex,Age,Patients number per hour,Arrival mode,Injury,Chief_complain,Mental,Pain,NRS_pain,...,BT,Saturation,KTAS_RN,Diagnosis in ED,Disposition,KTAS_expert,Error_group,Length of stay_min,KTAS duration_min,mistriage
0,2,2,71,3,3,2,right ocular pain,1,1,2,...,36.6,100,2,Corneal abrasion,1,4,2,86,"5,00",1
1,1,1,56,12,3,2,right forearm burn,1,1,2,...,36.5,NaN,4,"Burn of hand, firts degree dorsum",1,5,4,64,"3,95",1
2,2,1,68,8,2,2,"arm pain, Lt",1,1,2,...,36.6,98,4,"Fracture of surgical neck of humerus, closed",2,5,4,862,"1,00",1
3,1,2,71,8,1,1,ascites tapping,1,1,3,...,36.5,NaN,4,Alcoholic liver cirrhosis with ascites,1,5,6,108,"9,83",1
4,1,2,58,4,3,1,"distension, abd",1,1,3,...,36.5,NaN,4,Ascites,1,5,8,109,"6,60",1


In [3]:
ktas = pd.read_csv('00_raw/ktas_original.csv', encoding='latin-1', sep=';')

print("KTAS loaded successfully!")
print("Shape:", ktas.shape)
print("\nColumns:", ktas.columns.tolist())

KTAS loaded successfully!
Shape: (1267, 24)

Columns: ['Group', 'Sex', 'Age', 'Patients number per hour', 'Arrival mode', 'Injury', 'Chief_complain', 'Mental', 'Pain', 'NRS_pain', 'SBP', 'DBP', 'HR', 'RR', 'BT', 'Saturation', 'KTAS_RN', 'Diagnosis in ED', 'Disposition', 'KTAS_expert', 'Error_group', 'Length of stay_min', 'KTAS duration_min', 'mistriage']


In [4]:
ktas.head()

,Group,Sex,Age,Patients number per hour,Arrival mode,Injury,Chief_complain,Mental,Pain,NRS_pain,...,BT,Saturation,KTAS_RN,Diagnosis in ED,Disposition,KTAS_expert,Error_group,Length of stay_min,KTAS duration_min,mistriage
0,2,2,71,3,3,2,right ocular pain,1,1,2,...,36.6,100,2,Corneal abrasion,1,4,2,86,"5,00",1
1,1,1,56,12,3,2,right forearm burn,1,1,2,...,36.5,NaN,4,"Burn of hand, firts degree dorsum",1,5,4,64,"3,95",1
2,2,1,68,8,2,2,"arm pain, Lt",1,1,2,...,36.6,98,4,"Fracture of surgical neck of humerus, closed",2,5,4,862,"1,00",1
3,1,2,71,8,1,1,ascites tapping,1,1,3,...,36.5,NaN,4,Alcoholic liver cirrhosis with ascites,1,5,6,108,"9,83",1
4,1,2,58,4,3,1,"distension, abd",1,1,3,...,36.5,NaN,4,Ascites,1,5,8,109,"6,60",1


In [5]:
print("=== DATA TYPES ===")
print(ktas.dtypes)

print("\n=== MISSING VALUES ===")
print(ktas.isnull().sum())

print("\n=== KTAS_EXPERT TARGET DISTRIBUTION ===")
print(ktas['KTAS_expert'].value_counts())

print("\n=== KTAS_RN DISTRIBUTION ===")
print(ktas['KTAS_RN'].value_counts())

print("\n=== MISTRIAGE DISTRIBUTION ===")
print(ktas['mistriage'].value_counts())

=== DATA TYPES ===
Group                       int64
Sex                         int64
Age                         int64
Patients number per hour    int64
Arrival mode                int64
Injury                      int64
Chief_complain                str
Mental                      int64
Pain                        int64
NRS_pain                      str
SBP                           str
DBP                           str
HR                            str
RR                            str
BT                            str
Saturation                    str
KTAS_RN                     int64
Diagnosis in ED               str
Disposition                 int64
KTAS_expert                 int64
Error_group                 int64
Length of stay_min          int64
KTAS duration_min             str
mistriage                   int64
dtype: object

=== MISSING VALUES ===
Group                         0
Sex                           0
Age                           0
Patients number per hour      0

In [6]:
leakage_columns = [
    'Length of stay_min',
    'KTAS duration_min',
    'mistriage',
    'Error_group',
    'Disposition',
    'Diagnosis in ED',
    'KTAS_RN'
]

ktas = ktas.drop(columns=leakage_columns, errors='ignore')
print("Remaining columns:", ktas.columns.tolist())

Remaining columns: ['Group', 'Sex', 'Age', 'Patients number per hour', 'Arrival mode', 'Injury', 'Chief_complain', 'Mental', 'Pain', 'NRS_pain', 'SBP', 'DBP', 'HR', 'RR', 'BT', 'Saturation', 'KTAS_expert']


In [7]:
ktas['Pain'] = pd.to_numeric(ktas['Pain'], errors='coerce')
ktas['NRS_pain'] = pd.to_numeric(ktas['NRS_pain'], errors='coerce')

contradictions = ktas[(ktas['Pain'] == 0) & (ktas['NRS_pain'] > 0)]
print(f"Contradictory rows found: {len(contradictions)}")

ktas = ktas[~((ktas['Pain'] == 0) & (ktas['NRS_pain'] > 0))]
print(f"Rows remaining: {len(ktas)}")

Contradictory rows found: 0
Rows remaining: 1267


In [8]:
numeric_columns_to_fix = ['NRS_pain', 'SBP', 'DBP', 'HR', 'RR', 'BT', 'Saturation']
for col in numeric_columns_to_fix:
    if col in ktas.columns:
        ktas[col] = pd.to_numeric(ktas[col], errors='coerce')
print("Data types after conversion:")
print(ktas[numeric_columns_to_fix].dtypes)

Data types after conversion:
NRS_pain      float64
SBP           float64
DBP           float64
HR            float64
RR            float64
BT            float64
Saturation    float64
dtype: object


In [9]:
numeric_columns = ['SBP', 'DBP', 'HR', 'RR', 'BT', 'Age', 'NRS_pain', 'Saturation']

for col in numeric_columns:
    if col in ktas.columns:
        ktas[col] = ktas[col].fillna(ktas[col].median())

categorical_columns = ['Sex', 'Mental', 'Arrival mode', 'Injury']

for col in categorical_columns:
    if col in ktas.columns:
        ktas[col] = ktas[col].fillna(ktas[col].mode()[0])

ktas = ktas.dropna(subset=['KTAS_expert'])

print("Missing values after cleaning:")
print(ktas.isnull().sum())

Missing values after cleaning:
Group                       0
Sex                         0
Age                         0
Patients number per hour    0
Arrival mode                0
Injury                      0
Chief_complain              0
Mental                      0
Pain                        0
NRS_pain                    0
SBP                         0
DBP                         0
HR                          0
RR                          0
BT                          0
Saturation                  0
KTAS_expert                 0
dtype: int64


In [10]:
# Sex: 1 = Female, 2 = Male
if 'Sex' in ktas.columns:
    ktas['Sex'] = ktas['Sex'].map({1: 'Female', 2: 'Male'})

# Mental: 1 = Alert, 2 = Verbal, 3 = Pain, 4 = Unresponsive
if 'Mental' in ktas.columns:
    ktas['Mental'] = ktas['Mental'].map({
        1: 'Alert',
        2: 'Verbal',
        3: 'Pain',
        4: 'Unresponsive'
    })

# Injury: 1 = No, 2 = Yes
if 'Injury' in ktas.columns:
    ktas['Injury'] = ktas['Injury'].map({1: 'No', 2: 'Yes'})

# Pain: 0 = No, 1 = Yes
if 'Pain' in ktas.columns:
    ktas['Pain'] = ktas['Pain'].map({0: 'No', 1: 'Yes'})

# Arrival mode: combine rare categories (5, 6, 7) into 'Other'
if 'Arrival mode' in ktas.columns:
    ktas['Arrival mode'] = ktas['Arrival mode'].replace({5: 'Other', 6: 'Other', 7: 'Other'})

print("First 5 rows of categorical columns:")
print(ktas[['Sex', 'Mental', 'Injury', 'Pain']].head())

First 5 rows of categorical columns:
      Sex Mental Injury Pain
0    Male  Alert    Yes  Yes
1  Female  Alert    Yes  Yes
2  Female  Alert    Yes  Yes
3    Male  Alert     No  Yes
4    Male  Alert     No  Yes


In [11]:
ktas['KTAS_binary'] = ktas['KTAS_expert'].apply(
    lambda x: 'Emergency' if x in [1, 2, 3] else 'Non-Emergency'
)

print("Binary Target Distribution:")
print(ktas['KTAS_binary'].value_counts())

Binary Target Distribution:
KTAS_binary
Emergency        733
Non-Emergency    534
Name: count, dtype: int64


In [12]:
ktas.to_csv('01_interim/ktas_cleaned.csv', index=False)

print(f"Cleaned KTAS saved!")
print(f"Rows: {ktas.shape[0]}, Columns: {ktas.shape[1]}")

Cleaned KTAS saved!
Rows: 1267, Columns: 18


In [13]:
flow = pd.read_csv('00_raw/patient_flow_original.csv', encoding='latin-1')

print("Patient Flow loaded successfully!")
print("Shape:", flow.shape)
print("\nColumns:", flow.columns.tolist())

Patient Flow loaded successfully!
Shape: (60000, 14)

Columns: ['RunID', 'PatientID', 'ArrivalTime', 'Priority', 'DoctorID', 'DocServiceTime', 'MedServiceTime', 'DocWaitTime', 'MedWaitTime', 'TotalTimeInHospital', 'IsPeakArrival', 'Scenario', 'NumDoctors', 'PeakLimit']


In [14]:
flow.head()

,RunID,PatientID,ArrivalTime,Priority,DoctorID,DocServiceTime,MedServiceTime,DocWaitTime,MedWaitTime,TotalTimeInHospital,IsPeakArrival,Scenario,NumDoctors,PeakLimit
0,1,1,4,2,1,10,5,0,0,15,1,Baseline_2Doc_240Peak,2,240
1,1,2,7,1,2,15,2,0,0,17,1,Baseline_2Doc_240Peak,2,240
2,1,3,10,1,1,13,2,4,0,19,1,Baseline_2Doc_240Peak,2,240
3,1,4,13,1,1,18,3,14,0,35,1,Baseline_2Doc_240Peak,2,240
4,1,5,17,2,2,10,5,5,0,20,1,Baseline_2Doc_240Peak,2,240


In [15]:
print("-DATA TYPES ")
print(flow.dtypes)

print("\n-MISSING VALUES ")
print(flow.isnull().sum())

print("\n-SCENARIOS ")
print(flow['Scenario'].value_counts())

-DATA TYPES 


RunID                  int64
PatientID              int64
ArrivalTime            int64
Priority               int64
DoctorID               int64
DocServiceTime         int64
MedServiceTime         int64
DocWaitTime            int64
MedWaitTime            int64
TotalTimeInHospital    int64
IsPeakArrival          int64
Scenario                 str
NumDoctors             int64
PeakLimit              int64
dtype: object

-MISSING VALUES 
RunID                  0
PatientID              0
ArrivalTime            0
Priority               0
DoctorID               0
DocServiceTime         0
MedServiceTime         0
DocWaitTime            0
MedWaitTime            0
TotalTimeInHospital    0
IsPeakArrival          0
Scenario               0
NumDoctors             0
PeakLimit              0
dtype: int64

-SCENARIOS 
Scenario
Baseline_2Doc_240Peak    10000
Staffing_1Doc            10000
Staffing_3Doc            10000
Staffing_4Doc            10000
Peak_120Min              10000
Peak_360Min          

In [16]:
flow = flow.rename(columns={
    'ArrivalTime': 'arrival_min',
    'DocWaitTime': 'doc_wait_min',
    'MedWaitTime': 'med_wait_min',
    'DocServiceTime': 'doc_service_min',
    'MedServiceTime': 'med_service_min',
    'TotalTimeInHospital': 'total_time_min',
    'Priority': 'priority_level',
    'NumDoctors': 'num_doctors'
})

print("New columns:", flow.columns.tolist())

New columns: ['RunID', 'PatientID', 'arrival_min', 'priority_level', 'DoctorID', 'doc_service_min', 'med_service_min', 'doc_wait_min', 'med_wait_min', 'total_time_min', 'IsPeakArrival', 'Scenario', 'num_doctors', 'PeakLimit']


In [17]:
#Total waiting time
flow['total_wait_min'] = flow['doc_wait_min'] + flow['med_wait_min']

#Total service time
flow['total_service_min'] = flow['doc_service_min'] + flow['med_service_min']

# Total patient final time
flow['patient_final_time_min'] = flow['total_wait_min'] + flow['total_service_min']

print("New columns created:")
print(flow[['total_wait_min', 'total_service_min', 'patient_final_time_min', 'total_time_min']].head())

New columns created:
   total_wait_min  total_service_min  patient_final_time_min  total_time_min
0               0                 15                      15              15
1               0                 17                      17              17
2               4                 15                      19              19
3              14                 21                      35              35
4               5                 15                      20              20


In [18]:
# Calculating the difference
flow['difference'] = abs(flow['patient_final_time_min'] - flow['total_time_min'])

# Finding rows where difference is more than 0.01
mismatches = flow[flow['difference'] > 0.01]

print(f"Rows where calculated time != provided total: {len(mismatches)}")
print(f"Maximum difference found: {flow['difference'].max()}")

if len(mismatches) == 0:
    print("Calculated times match the total.")
else:
    print("Mismatches found.")

Rows where calculated time != provided total: 0
Maximum difference found: 0
Calculated times match the total.


In [19]:
flow['DateTime'] = pd.to_datetime('2024-01-01 08:00:00') + pd.to_timedelta(flow['arrival_min'], unit='m')

print(flow[['arrival_min', 'DateTime']].head())
# Assumming it starts at '2024-01-01 08:00:00'

   arrival_min            DateTime
0            4 2024-01-01 08:04:00
1            7 2024-01-01 08:07:00
2           10 2024-01-01 08:10:00
3           13 2024-01-01 08:13:00
4           17 2024-01-01 08:17:00


In [20]:
flow.to_csv('01_interim/patient_flow_cleaned.csv', index=False)

print(f"Cleaned Flow saved!")
print(f"Rows: {flow.shape[0]}")

Cleaned Flow saved!
Rows: 60000


In [21]:
# Grouping by Scenario and by hour
flow_hourly = flow.groupby(
    ['Scenario', pd.Grouper(key='DateTime', freq='h')]
).agg(
    patient_count=('PatientID', 'count'),
    avg_total_wait=('total_wait_min', 'mean'),
    avg_total_service=('total_service_min', 'mean'),
    avg_patient_final_time=('patient_final_time_min', 'mean'),
    max_patient_final_time=('patient_final_time_min', 'max')
).reset_index()

print("Time series aggregation created!")
print(flow_hourly.head(10))

Time series aggregation created!
                Scenario            DateTime  patient_count  avg_total_wait  \
0  Baseline_2Doc_240Peak 2024-01-01 08:00:00            677       18.793205   
1  Baseline_2Doc_240Peak 2024-01-01 09:00:00            690       63.655072   
2  Baseline_2Doc_240Peak 2024-01-01 10:00:00            695      104.172662   
3  Baseline_2Doc_240Peak 2024-01-01 11:00:00            702      131.789174   
4  Baseline_2Doc_240Peak 2024-01-01 12:00:00            553      149.887884   
5  Baseline_2Doc_240Peak 2024-01-01 13:00:00            546      175.230769   
6  Baseline_2Doc_240Peak 2024-01-01 14:00:00            548      199.742701   
7  Baseline_2Doc_240Peak 2024-01-01 15:00:00            554      198.772563   
8  Baseline_2Doc_240Peak 2024-01-01 16:00:00            560      210.092857   
9  Baseline_2Doc_240Peak 2024-01-01 17:00:00            548      226.883212   

   avg_total_service  avg_patient_final_time  max_patient_final_time  
0          16.290990      

In [22]:
flow_hourly.to_csv('02_processed/patient_flow_timeseries.csv', index=False)

print(f"Time series saved!")
print(f"Rows: {flow_hourly.shape[0]}")

Time series saved!
Rows: 111


In [23]:
# Reload the cleaned KTAS file
ktas_check = pd.read_csv('01_interim/ktas_cleaned.csv')

# Show missing values for each column
print("=== MISSING VALUES IN CLEANED KTAS ===")
missing = ktas_check.isnull().sum()
print(missing[missing > 0])

print(f"\nTotal missing values: {ktas_check.isnull().sum().sum()}")
print(f"Total rows: {ktas_check.shape[0]}")
print(f"Total columns: {ktas_check.shape[1]}")

=== MISSING VALUES IN CLEANED KTAS ===
Series([], dtype: int64)

Total missing values: 0
Total rows: 1267
Total columns: 18


In [24]:
ktas = pd.read_csv('01_interim/ktas_cleaned.csv')

# Filling every column that has empty cells
for col in ktas.columns:
    if ktas[col].isnull().sum() > 0:
        # Skip the target column
        if col == 'KTAS_expert':
            continue

        # If column has numbers, fill with median
        if pd.api.types.is_numeric_dtype(ktas[col]):
            median_val = ktas[col].median()
            if pd.isna(median_val):
                median_val = 0
            ktas[col] = ktas[col].fillna(median_val)
            print(f"Filled '{col}' with median: {median_val}")
        else:
            # If column has text, fill with most common value
            mode_val = ktas[col].mode()
            if len(mode_val) > 0:
                ktas[col] = ktas[col].fillna(mode_val[0])
                print(f"Filled '{col}' with mode: {mode_val[0]}")
            else:
                ktas[col] = ktas[col].fillna('Unknown')
                print(f"Filled '{col}' with 'Unknown'")

# Dropping rows where target is empty
ktas = ktas.dropna(subset=['KTAS_expert'])

# Showin result
print(f"\nDone! Total empty cells now: {ktas.isnull().sum().sum()}")


Done! Total empty cells now: 0


In [25]:
ktas.to_csv('01_interim/ktas_cleaned.csv', index=False)

print("File saved successfully!")

ktas_verify = pd.read_csv('01_interim/ktas_cleaned.csv')
print(f"Total empty cells in saved file: {ktas_verify.isnull().sum().sum()}")

if ktas_verify.isnull().sum().sum() == 0:
    print("No more empty cells.")
else:
    print("Still has empty cells.")

File saved successfully!
Total empty cells in saved file: 0
No more empty cells.


In [26]:
ktas_check = pd.read_csv('01_interim/ktas_cleaned.csv')
flow_check = pd.read_csv('01_interim/patient_flow_cleaned.csv')
ts_check = pd.read_csv('02_processed/patient_flow_timeseries.csv')

print(f"KTAS cleaned: {ktas_check.shape}")
print(f"Flow cleaned: {flow_check.shape}")
print(f"Time series: {ts_check.shape}")

# Check that KTAS has no missing values
missing = ktas_check.isnull().sum().sum()
print(f"\nKTAS missing cells: {missing}")

if missing == 0:
    print("All checks passed.")
else:
    print("KTAS still has missing values:")
    print(ktas_check.isnull().sum()[ktas_check.isnull().sum() > 0])

KTAS cleaned: (1267, 18)
Flow cleaned: (60000, 19)
Time series: (111, 7)

KTAS missing cells: 0
All checks passed.


In [27]:
import pandas as pd

ktas_raw = pd.read_csv('00_raw/ktas_original.csv', encoding='latin-1', sep=';')

print("=== ORIGINAL Sex values ===")
print(ktas_raw['Sex'].value_counts(dropna=False))

print("\n=== ORIGINAL Injury values ===")
print(ktas_raw['Injury'].value_counts(dropna=False))

print("\n=== ORIGINAL Mental values ===")
print(ktas_raw['Mental'].value_counts(dropna=False))

print("\n=== ORIGINAL Pain values ===")
print(ktas_raw['Pain'].value_counts(dropna=False))

=== ORIGINAL Sex values ===
Sex
2    661
1    606
Name: count, dtype: int64

=== ORIGINAL Injury values ===
Injury
1    1023
2     244
Name: count, dtype: int64

=== ORIGINAL Mental values ===
Mental
1    1187
2      39
3      28
4      13
Name: count, dtype: int64

=== ORIGINAL Pain values ===
Pain
1    714
0    553
Name: count, dtype: int64


In [28]:
import pandas as pd
import numpy as np

# Reload the raw data fresh
ktas = pd.read_csv('00_raw/ktas_original.csv', encoding='latin-1', sep=';')

print("Reloaded KTAS:", ktas.shape)

Reloaded KTAS: (1267, 24)


In [29]:
leakage_columns = [
    'Length of stay_min', 'KTAS duration_min',
    'mistriage', 'Error_group', 'Disposition',
    'Diagnosis in ED', 'KTAS_RN'
]
ktas = ktas.drop(columns=leakage_columns, errors='ignore')

print("Remaining columns:", ktas.columns.tolist())

Remaining columns: ['Group', 'Sex', 'Age', 'Patients number per hour', 'Arrival mode', 'Injury', 'Chief_complain', 'Mental', 'Pain', 'NRS_pain', 'SBP', 'DBP', 'HR', 'RR', 'BT', 'Saturation', 'KTAS_expert']


In [30]:
numeric_columns_to_fix = ['NRS_pain', 'SBP', 'DBP', 'HR', 'RR', 'BT', 'Saturation']

for col in numeric_columns_to_fix:
    if col in ktas.columns:
        ktas[col] = pd.to_numeric(ktas[col], errors='coerce')

print("Numeric conversion done.")
print(ktas[numeric_columns_to_fix].dtypes)

Numeric conversion done.
NRS_pain      float64
SBP           float64
DBP           float64
HR            float64
RR            float64
BT            float64
Saturation    float64
dtype: object


In [31]:
ktas['Pain'] = pd.to_numeric(ktas['Pain'], errors='coerce')
ktas['NRS_pain'] = pd.to_numeric(ktas['NRS_pain'], errors='coerce')

contradictions = ktas[(ktas['Pain'] == 0) & (ktas['NRS_pain'] > 0)]
print(f"Contradictions found: {len(contradictions)}")

ktas = ktas[~((ktas['Pain'] == 0) & (ktas['NRS_pain'] > 0))]
print(f"Rows remaining: {len(ktas)}")

Contradictions found: 0
Rows remaining: 1267


In [32]:
# Map categories FIRST (before filling missing values)
ktas['Sex'] = ktas['Sex'].map({1: 'Female', 2: 'Male'})
ktas['Mental'] = ktas['Mental'].map({
    1: 'Alert', 2: 'Verbal', 3: 'Pain', 4: 'Unresponsive'
})
ktas['Injury'] = ktas['Injury'].map({1: 'No', 2: 'Yes'})
ktas['Pain'] = ktas['Pain'].map({0: 'No', 1: 'Yes'})
ktas['Arrival mode'] = ktas['Arrival mode'].replace({
    5: 'Other', 6: 'Other', 7: 'Other'
})

print("Categories mapped.")
print(ktas[['Sex', 'Mental', 'Injury', 'Pain']].head())

Categories mapped.
      Sex Mental Injury Pain
0    Male  Alert    Yes  Yes
1  Female  Alert    Yes  Yes
2  Female  Alert    Yes  Yes
3    Male  Alert     No  Yes
4    Male  Alert     No  Yes


In [33]:
# Numeric columns: fill with median
numeric_cols_to_fill = ['SBP', 'DBP', 'HR', 'RR', 'BT', 'Age', 'NRS_pain', 'Saturation']
for col in numeric_cols_to_fill:
    if col in ktas.columns:
        ktas[col] = ktas[col].fillna(ktas[col].median())

# Categorical columns: fill with 'Unknown'
for col in ['Sex', 'Mental', 'Injury', 'Pain']:
    ktas[col] = ktas[col].fillna('Unknown')

# Drop rows without target
ktas = ktas.dropna(subset=['KTAS_expert'])

print("Missing values remaining:")
print(ktas.isnull().sum().sum())

Missing values remaining:
0


In [34]:
ktas['KTAS_binary'] = ktas['KTAS_expert'].apply(
    lambda x: 'Emergency' if x in [1, 2, 3] else 'Non-Emergency'
)
print(ktas['KTAS_binary'].value_counts())

KTAS_binary
Emergency        733
Non-Emergency    534
Name: count, dtype: int64


In [35]:
print(" Sex ")
print(ktas['Sex'].value_counts(dropna=False))

print("\n Mental ")
print(ktas['Mental'].value_counts(dropna=False))

print("\n Injury ")
print(ktas['Injury'].value_counts(dropna=False))

print("\n Pain ")
print(ktas['Pain'].value_counts(dropna=False))

 Sex 


Sex
Male      661
Female    606
Name: count, dtype: int64

 Mental 
Mental
Alert           1187
Verbal            39
Pain              28
Unresponsive      13
Name: count, dtype: int64

 Injury 
Injury
No     1023
Yes     244
Name: count, dtype: int64

 Pain 
Pain
Yes    714
No     553
Name: count, dtype: int64


In [36]:
ktas.to_csv('01_interim/ktas_cleaned.csv', index=False)

print(f" Corrected KTAS saved!")
print(f"Rows: {ktas.shape[0]}, Columns: {ktas.shape[1]}")

 Corrected KTAS saved!
Rows: 1267, Columns: 18


In [37]:
import pandas as pd
import numpy as np
import re
import os

os.makedirs('01_interim', exist_ok=True)
os.makedirs('02_processed', exist_ok=True)

print("\n[1/2] Cleaning KTAS dataset...")

ktas = pd.read_csv('00_raw/ktas_original.csv', encoding='latin-1', sep=';')
print(f"  Loaded: {ktas.shape[0]} rows, {ktas.shape[1]} columns")

numeric_columns_to_fix = ['NRS_pain', 'SBP', 'DBP', 'HR', 'RR', 'BT', 'Saturation']
for col in numeric_columns_to_fix:
    if col in ktas.columns:
        ktas[col] = pd.to_numeric(ktas[col], errors='coerce')

ktas['Pain'] = pd.to_numeric(ktas['Pain'], errors='coerce')
ktas['NRS_pain'] = pd.to_numeric(ktas['NRS_pain'], errors='coerce')
ktas = ktas[~((ktas['Pain'] == 0) & (ktas['NRS_pain'] > 0))]

# Sex, Mental, Injury, Pain remain as 1, 2, 3, 0

numeric_cols_to_fill = ['SBP', 'DBP', 'HR', 'RR', 'BT', 'Age', 'NRS_pain', 'Saturation']
for col in numeric_cols_to_fill:
    if col in ktas.columns:
        ktas[col] = ktas[col].fillna(ktas[col].median())

for col in ['Sex', 'Mental', 'Injury', 'Pain', 'Arrival mode']:
    if col in ktas.columns:
        ktas[col] = ktas[col].fillna(ktas[col].mode()[0])

ktas = ktas.dropna(subset=['KTAS_expert'])

# 1 = Emergency, 0 = Non-Emergency
ktas['KTAS_binary'] = ktas['KTAS_expert'].apply(
    lambda x: 1 if x in [1, 2, 3] else 0
)

def clean_complaint(text):
    if pd.isna(text):
        return 'unknown'
    text = str(text).strip()
    # Remove ?? symbols
    text = re.sub(r'\?+', '', text)
    # Expand abbreviations
    abbreviations = {
        r'\bHA\b': 'headache',
        r'\bLBP\b': 'low back pain',
        r'\bCP\b': 'chest pain',
        r'\bSOB\b': 'shortness of breath',
        r'\bN/V\b': 'nausea and vomiting',
        r'\bF/C\b': 'fever and chills',
        r'\bC/S\b': 'chills and sweating',
        r'\babd\b': 'abdominal',
        r'\bDZ\b': 'dizziness',
        r'\bLOC\b': 'loss of consciousness',
        r'\bFB\b': 'foreign body',
        r'\bant\.': 'anterior',
        r'\brt\.': 'right',
        r'\blt\.': 'left',
        r'\bRt\.': 'right',
        r'\bLt\.': 'left',
        r'\bRLQ\b': 'right lower quadrant',
        r'\bLLQ\b': 'left lower quadrant',
        r'\bRUQ\b': 'right upper quadrant',
        r'\bLUQ\b': 'left upper quadrant',
        r'\bH-\b': 'headache -',
    }
    for abbr, full in abbreviations.items():
        text = re.sub(abbr, full, text, flags=re.IGNORECASE)
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    # Remove leading/trailing slashes and dashes
    text = text.strip('/- ')
    if text == '':
        return 'unknown'
    return text

ktas['Chief_complain'] = ktas['Chief_complain'].apply(clean_complaint)

ktas.to_csv('01_interim/ktas_cleaned.csv', index=False)
print(f"  Saved: {ktas.shape[0]} rows, {ktas.shape[1]} columns")

print("\n[2/2] Cleaning Patient Flow dataset...")

flow = pd.read_csv('00_raw/patient_flow_original.csv', encoding='latin-1')
print(f"  Loaded: {flow.shape[0]} rows, {flow.shape[1]} columns")

# The original file has 14 columns. Keep them all.
# Remove any derived columns if they exist.
derived_columns_to_remove = [
    'total_wait_min', 'total_service_min',
    'patient_final_time_min', 'DateTime', 'difference'
]
flow = flow.drop(columns=derived_columns_to_remove, errors='ignore')

print(f"  Final columns: {flow.shape[1]}")
print(f"  Column names: {flow.columns.tolist()}")

flow.to_csv('01_interim/patient_flow_cleaned.csv', index=False)
print(f"  Saved: {flow.shape[0]} rows, {flow.shape[1]} columns")

print("\n" + "=" * 60)

ktas_v = pd.read_csv('01_interim/ktas_cleaned.csv')
flow_v = pd.read_csv('01_interim/patient_flow_cleaned.csv')

print(f"\nKTAS: {ktas_v.shape}")
print(f"KTAS columns: {ktas_v.columns.tolist()}")
print(f"KTAS missing values: {ktas_v.isnull().sum().sum()}")

print(f"\nKTAS Sex unique values: {sorted(ktas_v['Sex'].unique())}")
print(f"KTAS Mental unique values: {sorted(ktas_v['Mental'].unique())}")
print(f"KTAS Injury unique values: {sorted(ktas_v['Injury'].unique())}")
print(f"KTAS Pain unique values: {sorted(ktas_v['Pain'].unique())}")
print(f"KTAS_binary unique values: {sorted(ktas_v['KTAS_binary'].unique())}")

print(f"\nFlow: {flow_v.shape}")
print(f"Flow columns: {flow_v.columns.tolist()}")

print(f"\nSample cleaned Chief_complain:")
print(ktas_v['Chief_complain'].head(5).tolist())

print("\nDONE.")


[1/2] Cleaning KTAS dataset...
  Loaded: 1267 rows, 24 columns
  Saved: 1267 rows, 25 columns

[2/2] Cleaning Patient Flow dataset...


  Loaded: 60000 rows, 14 columns


  Final columns: 14
  Column names: ['RunID', 'PatientID', 'ArrivalTime', 'Priority', 'DoctorID', 'DocServiceTime', 'MedServiceTime', 'DocWaitTime', 'MedWaitTime', 'TotalTimeInHospital', 'IsPeakArrival', 'Scenario', 'NumDoctors', 'PeakLimit']


  Saved: 60000 rows, 14 columns




KTAS: (1267, 25)
KTAS columns: ['Group', 'Sex', 'Age', 'Patients number per hour', 'Arrival mode', 'Injury', 'Chief_complain', 'Mental', 'Pain', 'NRS_pain', 'SBP', 'DBP', 'HR', 'RR', 'BT', 'Saturation', 'KTAS_RN', 'Diagnosis in ED', 'Disposition', 'KTAS_expert', 'Error_group', 'Length of stay_min', 'KTAS duration_min', 'mistriage', 'KTAS_binary']
KTAS missing values: 2



KTAS Sex unique values: [np.int64(1), np.int64(2)]
KTAS Mental unique values: [np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
KTAS Injury unique values: [np.int64(1), np.int64(2)]
KTAS Pain unique values: [np.int64(0), np.int64(1)]
KTAS_binary unique values: [np.int64(0), np.int64(1)]

Flow: (60000, 14)
Flow columns: ['RunID', 'PatientID', 'ArrivalTime', 'Priority', 'DoctorID', 'DocServiceTime', 'MedServiceTime', 'DocWaitTime', 'MedWaitTime', 'TotalTimeInHospital', 'IsPeakArrival', 'Scenario', 'NumDoctors', 'PeakLimit']

Sample cleaned Chief_complain:
['right ocular pain', 'right forearm burn', 'arm pain, Lt', 'ascites tapping', 'distension, abdominal']

DONE.


In [38]:
# Removes all symbols and expands all abbreviations

import pandas as pd
import re
import os

os.makedirs('01_interim', exist_ok=True)

# This dictionary contains EVERY abbreviation found in the dataset
# and what it should become in full English.

abbreviation_map = {

    r'\brt\.?\b': 'right',
    r'\blt\.?\b': 'left',
    r'\bbilat\.?\b': 'bilateral',
    r'\bant\.?\b': 'anterior',
    r'\bpost\.?\b': 'posterior',
    r'\bext\.?\b': 'extremity',
    r'\bperi-?umbilical\b': 'periumbilical',

    r'\babd\.?\b': 'abdominal',
    r'\bHA\b': 'headache',
    r'\bH\b': 'headache',
    r'\bLBP\b': 'low back pain',
    r'\bCP\b': 'chest pain',
    r'\bSOB\b': 'shortness of breath',
    r'\bTMJ\b': 'temporomandibular joint',
    r'\bFB\b': 'foreign body',
    r'\bPTx\b': 'pneumothorax',
    r'\bAVF\b': 'arteriovenous fistula',
    r'\bPPROM\b': 'preterm premature rupture of membranes',
    r'\bBST\b': 'blood sugar test',
    r'\bBSL\b': 'blood sugar level',
    r'\bHb\b': 'hemoglobin',
    r'\bDI\b': 'drug intoxication',
    r'\bLOC\b': 'loss of consciousness',
    r'\bDZ\b': 'dizziness',
    r'\bN/V\b': 'nausea and vomiting',
    r'\bF/C\b': 'fever and chills',
    r'\bC/S\b': 'chills and sweating',
    r'\bw/u\b': 'workup',
    r'\bSx\b': 'symptom',
    r'\bpo\b': 'by mouth',
    r'\bIV\b': 'intravenous',

    r'\bRLQ\b': 'right lower quadrant',
    r'\bLLQ\b': 'left lower quadrant',
    r'\bRUQ\b': 'right upper quadrant',
    r'\bLUQ\b': 'left upper quadrant',

    r'\bSOB\b': 'shortness of breath',
    r'\bLOC\b': 'loss of consciousness',
    r'\bF/C\b': 'fever and chills',
    r'\bC/S\b': 'chills and sweating',

    r'\bO\.?T\.?\b': 'operating theatre',
    r'\bICU\b': 'intensive care unit',
    r'\bED\b': 'emergency department',
    r'\bGI\b': 'gastrointestinal',
    r'\bGU\b': 'genitourinary',
    r'\bCVA\b': 'cerebrovascular accident',
    r'\bCNS\b': 'central nervous system',

    r'\bBT\b': 'body temperature',
    r'\bBP\b': 'blood pressure',
    r'\bSBP\b': 'systolic blood pressure',
    r'\bDBP\b': 'diastolic blood pressure',
    r'\bHR\b': 'heart rate',
    r'\bRR\b': 'respiratory rate',
    r'\bSpO2\b': 'oxygen saturation',
    r'\bNRS\b': 'numeric rating scale',

    r'\bPalp\.?\b': 'palpitation',
    r'\bSz\b': 'seizure',
    r'\bCONV\b': 'convulsion',
    r'\bAMS\b': 'altered mental status',
    r'\bGTC\b': 'generalised tonic-clonic seizure',
}

def clean_text(text):
    if pd.isna(text):
        return 'unknown'

    text = str(text).strip()

    text = re.sub(r'\?+', '', text)

    text = text.replace('/', ' and ')

    text = text.replace('-', ' ')

    text = text.replace(',', ' ')

    text = text.replace('+', ' ')
    text = text.replace('(', ' ')
    text = text.replace(')', ' ')

    text = text.replace('.', ' ')
    text = text.replace(':', ' ')

    for abbr, full in abbreviation_map.items():
        text = re.sub(abbr, full, text, flags=re.IGNORECASE)

    spelling_fixes = {
        r'\bAbd\b': 'abdominal',
        r'\babd\b': 'abdominal',
        r'\bRt\b': 'right',
        r'\bLt\b': 'left',
        r'\bResp\b': 'respiratory',
        r'\bCardiac\b': 'cardiac',
        r'\bNeuro\b': 'neurological',
    }
    for wrong, right in spelling_fixes.items():
        text = re.sub(wrong, right, text, flags=re.IGNORECASE)

    text = re.sub(r'\s+', ' ', text).strip()

    text = text.lower()

    if text == '':
        return 'unknown'

    return text

print("\n[1/2] Cleaning KTAS Chief_complain...")

ktas = pd.read_csv('00_raw/ktas_original.csv', encoding='latin-1', sep=';')
print(f"  Loaded: {ktas.shape[0]} rows")

# Show BEFORE
print("\n  BEFORE (first 10 rows):")
for i, text in enumerate(ktas['Chief_complain'].head(10)):
    print(f"    {i+1}. {text}")

# Apply cleaning
ktas['Chief_complain_cleaned'] = ktas['Chief_complain'].apply(clean_text)

# Show AFTER
print("\n  AFTER (first 10 rows):")
for i, text in enumerate(ktas['Chief_complain_cleaned'].head(10)):
    print(f"    {i+1}. {text}")

ktas.to_csv('01_interim/ktas_cleaned.csv', index=False)
print(f"\n  Saved: 01_interim/ktas_cleaned.csv")

print("\n[2/2] Cleaning Patient Flow...")

flow = pd.read_csv('00_raw/patient_flow_original.csv', encoding='latin-1')
print(f"  Loaded: {flow.shape[0]} rows")

flow.to_csv('01_interim/patient_flow_cleaned.csv', index=False)
print(f"  Saved: 01_interim/patient_flow_cleaned.csv")

print("\n" + "=" * 60)
print("SAMPLE TRANSFORMATIONS")

samples = [
    ("ant. chest pain", "anterior chest pain"),
    ("abd pain", "abdominal pain"),
    ("F/C, C/S", "fever and chills chills and sweating"),
    ("HA - Headache", "headache headache"),
    ("Rt. flank pain", "right flank pain"),
    ("Lt. hip joint pain", "left hip joint pain"),
    ("TMJ pain", "temporomandibular joint pain"),
    ("DI, drug intoxication", "drug intoxication drug intoxication"),
    ("for A/tapping", "for a and tapping"),
    ("RLQ pain", "right lower quadrant pain"),
    ("SOB", "shortness of breath"),
    ("LOC - Loss of consciousness", "loss of consciousness loss of consciousness"),
]

print(f"\n{'ORIGINAL':<40} → {'CLEANED':<40}")
for original, expected in samples:
    cleaned = clean_text(original)
    print(f"{original:<40} → {cleaned:<40}")

print("\n" + "=" * 60)
print("ALL DONE")
print("\nFiles saved:")
print("  01_interim/ktas_cleaned.csv")
print("  01_interim/patient_flow_cleaned.csv")
print("\nThe Chief_complain_cleaned column has been added.")


[1/2] Cleaning KTAS Chief_complain...
  Loaded: 1267 rows

  BEFORE (first 10 rows):
    1. right ocular pain
    2. right forearm burn
    3. arm pain, Lt
    4. ascites tapping
    5. distension, abd
    6. fever
    7. With chest discomfort
    8. pain, chest
    9. LBP - Low back pain
    10. Eczema, Eyelid

  AFTER (first 10 rows):
    1. right ocular pain
    2. right forearm burn
    3. arm pain left
    4. ascites tapping
    5. distension abdominal
    6. fever
    7. with chest discomfort
    8. pain chest
    9. low back pain low back pain
    10. eczema eyelid

  Saved: 01_interim/ktas_cleaned.csv

[2/2] Cleaning Patient Flow...


  Loaded: 60000 rows


  Saved: 01_interim/patient_flow_cleaned.csv

SAMPLE TRANSFORMATIONS

ORIGINAL                                 → CLEANED                                 
ant. chest pain                          → anterior chest pain                     
abd pain                                 → abdominal pain                          
F/C, C/S                                 → f and c c and s                         
HA - Headache                            → headache headache                       
Rt. flank pain                           → right flank pain                        
Lt. hip joint pain                       → left hip joint pain                     
TMJ pain                                 → temporomandibular joint pain            
DI, drug intoxication                    → drug intoxication drug intoxication     
for A/tapping                            → for a and tapping                       
RLQ pain                                 → right lower quadrant pain               
SOB   

In [39]:
import pandas as pd
import numpy as np

# Try this first. If it works, great!
try:
    ktas = pd.read_csv('00_raw/ktas_original.csv', encoding='utf-8', sep=';')
    print("Successfully read with utf-8 encoding.")
except UnicodeDecodeError:
    print("utf-8 failed, trying ISO-8859-9 (Turkish encoding)...")
    ktas = pd.read_csv('00_raw/ktas_original.csv', encoding='ISO-8859-9', sep=';')
    print("Successfully read with ISO-8859-9 encoding.")

# Now, the 'NRS_pain' column should contain only numbers and missing values.
# Let's ensure it's numeric.
ktas['NRS_pain'] = pd.to_numeric(ktas['NRS_pain'], errors='coerce')

# Now you can fill the missing values with the median.
ktas['NRS_pain'] = ktas['NRS_pain'].fillna(ktas['NRS_pain'].median())

# Check the result to confirm the fix
print("\nNRS_pain unique values after re-loading with correct encoding:")
print(sorted(ktas['NRS_pain'].dropna().unique()))

utf-8 failed, trying ISO-8859-9 (Turkish encoding)...
Successfully read with ISO-8859-9 encoding.

NRS_pain unique values after re-loading with correct encoding:
[np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0), np.float64(6.0), np.float64(7.0), np.float64(8.0), np.float64(9.0), np.float64(10.0)]


In [40]:
import pandas as pd

encodings_to_try = ['utf-8', 'ISO-8859-9', 'latin-1', 'cp1252', 'utf-8-sig']

for enc in encodings_to_try:
    try:
        df = pd.read_csv('00_raw/ktas_original.csv', encoding=enc, sep=';')
        # Check the NRS_pain column
        garbage_count = df['NRS_pain'].astype(str).str.contains('#BO', na=False).sum()
        print(f"Encoding '{enc}': SUCCESS - Found {garbage_count} garbage values")
    except Exception as e:
        print(f"Encoding '{enc}': FAILED - {e}")

Encoding 'utf-8': FAILED - 'utf-8' codec can't decode byte 0xde in position 3: invalid continuation byte
Encoding 'ISO-8859-9': SUCCESS - Found 556 garbage values
Encoding 'latin-1': SUCCESS - Found 556 garbage values
Encoding 'cp1252': SUCCESS - Found 556 garbage values
Encoding 'utf-8-sig': FAILED - 'utf-8' codec can't decode byte 0xde in position 3270: invalid continuation byte


In [41]:
import pandas as pd
import numpy as np

ktas = pd.read_csv('00_raw/ktas_original.csv', encoding='ISO-8859-9', sep=';')

print("Raw unique values in NRS_pain (first 20):")
print(ktas['NRS_pain'].astype(str).unique()[:20])

# Method 1: Direct replacement of the specific garbage string
ktas['NRS_pain'] = ktas['NRS_pain'].replace('#BOÃž!', np.nan)

# Method 2: Also catch any variations (safer)
ktas['NRS_pain'] = ktas['NRS_pain'].astype(str).str.replace(r'#BO.*', '', regex=True)
ktas['NRS_pain'] = ktas['NRS_pain'].replace('', np.nan)

# Method 3: The most robust — force everything to numeric
# Anything that can't be converted becomes NaN
ktas['NRS_pain'] = pd.to_numeric(ktas['NRS_pain'], errors='coerce')

print("\nCleaned unique values in NRS_pain:")
print(sorted(ktas['NRS_pain'].dropna().unique()))

print(f"\nMissing values in NRS_pain: {ktas['NRS_pain'].isnull().sum()}")
print(f"Total rows: {len(ktas)}")

Raw unique values in NRS_pain (first 20):
<StringArray>
['2', '3', '4', '5', '6', '8', '9', '#BOŞ!', '1', '7', '10']
Length: 11, dtype: str

Cleaned unique values in NRS_pain:
[np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0), np.float64(6.0), np.float64(7.0), np.float64(8.0), np.float64(9.0), np.float64(10.0)]

Missing values in NRS_pain: 556
Total rows: 1267


In [42]:
ktas['NRS_pain'] = pd.to_numeric(ktas['NRS_pain'], errors='coerce')

In [43]:
import pandas as pd

ktas = pd.read_csv('00_raw/ktas_original.csv', encoding='ISO-8859-9', sep=';')

print("All column names in the KTAS dataset:")
for i, col in enumerate(ktas.columns):
    print(f"  {i}: '{col}'")

# Also print the first 3 rows to see the data
print("\nFirst 3 rows:")
print(ktas.head(3))

All column names in the KTAS dataset:
  0: 'Group'
  1: 'Sex'
  2: 'Age'
  3: 'Patients number per hour'
  4: 'Arrival mode'
  5: 'Injury'
  6: 'Chief_complain'
  7: 'Mental'
  8: 'Pain'
  9: 'NRS_pain'
  10: 'SBP'
  11: 'DBP'
  12: 'HR'
  13: 'RR'
  14: 'BT'
  15: 'Saturation'
  16: 'KTAS_RN'
  17: 'Diagnosis in ED'
  18: 'Disposition'
  19: 'KTAS_expert'
  20: 'Error_group'
  21: 'Length of stay_min'
  22: 'KTAS duration_min'
  23: 'mistriage'

First 3 rows:
   Group  Sex  Age  Patients number per hour  Arrival mode  Injury  \
0      2    2   71                         3             3       2   
1      1    1   56                        12             3       2   
2      2    1   68                         8             2       2   

       Chief_complain  Mental  Pain NRS_pain  ...    BT Saturation KTAS_RN  \
0   right ocular pain       1     1        2  ...  36.6        100       2   
1  right forearm burn       1     1        2  ...  36.5        NaN       4   
2        arm pain, L

In [44]:
import pandas as pd
import numpy as np
import os

os.makedirs('00_raw', exist_ok=True)
os.makedirs('01_interim', exist_ok=True)

ktas = pd.read_csv('00_raw/ktas_original.csv', encoding='ISO-8859-9', sep=';')
print(f"\nLoaded: {ktas.shape[0]} rows, {ktas.shape[1]} columns")

numeric_cols = ['SBP', 'DBP', 'HR', 'RR', 'BT', 'Age', 'NRS_pain', 'Saturation']
for col in numeric_cols:
    if col in ktas.columns:
        original_nulls = ktas[col].isnull().sum()
        ktas[col] = pd.to_numeric(ktas[col], errors='coerce')
        new_nulls = ktas[col].isnull().sum()
        print(f"  '{col}': {new_nulls - original_nulls} garbage values converted to NaN")

leakage_columns = [
    'Length of stay_min',     # Only known after discharge
    'KTAS duration_min',      # Only known after triage
    'mistriage',              # Measures error - only known after comparison
    'Disposition',            # Outcome - only known after treatment
    'Diagnosis in ED',        # Final diagnosis - only known after workup
    'Error_group',            # Measures error - leakage!
    'KTAS_RN',                # Nurse's prediction - leakage!
    'Patients number per hour' # May be calculated post-hoc
]
ktas = ktas.drop(columns=leakage_columns, errors='ignore')
print(f"\nRemoved leakage columns. Shape: {ktas.shape}")

ktas['Pain'] = pd.to_numeric(ktas['Pain'], errors='coerce')
contradictions = ktas[(ktas['Pain'] == 0) & (ktas['NRS_pain'] > 0)]
print(f"Removed {len(contradictions)} contradictory pain records.")
ktas = ktas[~((ktas['Pain'] == 0) & (ktas['NRS_pain'] > 0))]

print("\nFilling missing values:")
for col in ['SBP', 'DBP', 'HR', 'RR', 'BT', 'Age', 'NRS_pain', 'Saturation']:
    if col in ktas.columns:
        median_val = ktas[col].median()
        nulls_before = ktas[col].isnull().sum()
        ktas[col] = ktas[col].fillna(median_val)
        print(f"  '{col}': filled {nulls_before} nulls with median ({median_val})")

for col in ['Sex', 'Mental', 'Injury', 'Pain', 'Arrival mode']:
    if col in ktas.columns:
        mode_val = ktas[col].mode()[0]
        nulls_before = ktas[col].isnull().sum()
        ktas[col] = ktas[col].fillna(mode_val)
        print(f"  '{col}': filled {nulls_before} nulls with mode ({mode_val})")

# The target is 'KTAS_expert' - the expert-assigned triage level
ktas = ktas.dropna(subset=['KTAS_expert'])
print(f"\nDropped rows with missing KTAS_expert. Final shape: {ktas.shape}")

# Create KTAS_binary from KTAS_expert
# 1-3 = Emergency, 4-5 = Non-Emergency
ktas['KTAS_binary'] = ktas['KTAS_expert'].apply(
    lambda x: 'Emergency' if x in [1, 2, 3] else 'Non-Emergency'
)
print(f"\nKTAS_binary distribution:")
print(ktas['KTAS_binary'].value_counts())

print(f"\nKTAS_expert distribution:")
print(ktas['KTAS_expert'].value_counts().sort_index())

print("\n" + "=" * 60)

print(f"\nTotal missing values: {ktas.isnull().sum().sum()}")
print(f"NRS_pain unique values: {sorted(ktas['NRS_pain'].unique())}")
print(f"KTAS_expert unique values: {sorted(ktas['KTAS_expert'].unique())}")
print(f"KTAS_binary unique values: {ktas['KTAS_binary'].unique()}")

print(f"\nFinal columns: {ktas.columns.tolist()}")
print(f"Final shape: {ktas.shape}")

ktas.to_csv('01_interim/ktas_cleaned.csv', index=False)
print(f"\nSaved to 01_interim/ktas_cleaned.csv")
print(f"   Final shape: {ktas.shape}")


Loaded: 1267 rows, 24 columns
  'SBP': 25 garbage values converted to NaN
  'DBP': 29 garbage values converted to NaN
  'HR': 20 garbage values converted to NaN
  'RR': 22 garbage values converted to NaN
  'BT': 18 garbage values converted to NaN
  'Age': 0 garbage values converted to NaN
  'NRS_pain': 556 garbage values converted to NaN
  'Saturation': 9 garbage values converted to NaN

Removed leakage columns. Shape: (1267, 16)
Removed 0 contradictory pain records.

Filling missing values:
  'SBP': filled 25 nulls with median (130.0)
  'DBP': filled 29 nulls with median (80.0)
  'HR': filled 20 nulls with median (82.0)
  'RR': filled 22 nulls with median (20.0)
  'BT': filled 18 nulls with median (36.5)
  'Age': filled 0 nulls with median (57.0)
  'NRS_pain': filled 556 nulls with median (4.0)
  'Saturation': filled 697 nulls with median (98.0)
  'Sex': filled 0 nulls with mode (2)
  'Mental': filled 0 nulls with mode (1)
  'Injury': filled 0 nulls with mode (1)
  'Pain': filled 0 n

In [45]:
import pandas as pd
import os

# all 24 columms

ktas_original = pd.read_csv('00_raw/ktas_original.csv', encoding='ISO-8859-9', sep=';')

print("Original shape:", ktas_original.shape)
print("Original columns:", ktas_original.columns.tolist())

# CLEANED KTAS file

ktas_cleaned = pd.read_csv('01_interim/ktas_cleaned.csv')

print("\nCleaned shape:", ktas_cleaned.shape)
print("Cleaned columns:", ktas_cleaned.columns.tolist())

#columns missing from CLEANED

missing_columns = [col for col in ktas_original.columns if col not in ktas_cleaned.columns]

print("\nMissing columns that need to be added back:")
for col in missing_columns:
    print("  -", col)

#missing columns back from the original file

for col in missing_columns:
    ktas_cleaned[col] = ktas_original[col].values

print("\nNew shape after adding columns:", ktas_cleaned.shape)
print("All columns now:", ktas_cleaned.columns.tolist())

output_path = '01_interim/ktas_cleaned.csv'

try:
    ktas_cleaned.to_csv(output_path, index=False)
    print(f"\n Saved successfully to {output_path}")
    print(f"   Final shape: {ktas_cleaned.shape}")
except PermissionError:
    print(f"\n STILL LOCKED: Close the file '{output_path}' in Excel/Notepad,")
    print("   then re-run this cell.")

Original shape: (1267, 24)
Original columns: ['Group', 'Sex', 'Age', 'Patients number per hour', 'Arrival mode', 'Injury', 'Chief_complain', 'Mental', 'Pain', 'NRS_pain', 'SBP', 'DBP', 'HR', 'RR', 'BT', 'Saturation', 'KTAS_RN', 'Diagnosis in ED', 'Disposition', 'KTAS_expert', 'Error_group', 'Length of stay_min', 'KTAS duration_min', 'mistriage']

Cleaned shape: (1267, 17)
Cleaned columns: ['Group', 'Sex', 'Age', 'Arrival mode', 'Injury', 'Chief_complain', 'Mental', 'Pain', 'NRS_pain', 'SBP', 'DBP', 'HR', 'RR', 'BT', 'Saturation', 'KTAS_expert', 'KTAS_binary']

Missing columns that need to be added back:
  - Patients number per hour
  - KTAS_RN
  - Diagnosis in ED
  - Disposition
  - Error_group
  - Length of stay_min
  - KTAS duration_min
  - mistriage

New shape after adding columns: (1267, 25)
All columns now: ['Group', 'Sex', 'Age', 'Arrival mode', 'Injury', 'Chief_complain', 'Mental', 'Pain', 'NRS_pain', 'SBP', 'DBP', 'HR', 'RR', 'BT', 'Saturation', 'KTAS_expert', 'KTAS_binary', 'P